# TraceLog Real RAG Benchmark

This notebook follows the constraints in `docs/eval/test_strategy.md` and runs dataset generation, retrieval evaluation, diagnosis evaluation, and reporting in one place.

The primary benchmark compares `Standard Log + RAG + Code` against `TraceLog + RAG + Code`. The `logs-only` comparison is treated as an ablation.

## How This Benchmark Is Built

This experiment runs on a synthetic dataset designed to resemble operational incidents. The key rule is `same codebase, different instrumentation`. Each incident is generated from the same scenario code, and only the output path changes.

1. `standard logging` path: emits plain text logs.
2. `TraceLog` path: emits TraceLog JSON dumps, then the Aggregator reconstructs them into unified Trace-DSL.

Historical incidents are indexed into the vector store, and query incidents are kept as holdout cases. RAG searches the historical corpus using the query incident, the Analyst diagnoses the current incident, and the Judge compares that diagnosis against sealed truth for root cause, surface error, evidence grounding, and actionability.

## Dataset Structure

The current dataset contains three scenario families.

- `ecommerce_bulk_checkout`: an upstream state bug such as coupon normalization appears later as a payment failure
- `warehouse_sync_reservation`: async/state issues such as snapshot version leaks or dedupe collisions appear as timeouts
- `api_gateway_audit`: tenant, token, and profile mismatches appear as permission failures

Each incident produces the following artifacts.

- `standard.log`: baseline logs
- `tracelog_dump.jsonl`: JSON dumps emitted by the TraceLog exporter
- `aggregated_trace.log`: unified trace produced by the Aggregator after span reconstruction
- `truth/*.json`: sealed truth hidden from the Analyst and shown only to the Judge

The split is divided into `historical` and `query`. Historical incidents form the retrieval corpus, and query incidents are the actual evaluation targets.

## Evaluation Structure

This notebook shows two layers of comparison.

- Primary benchmark: `Standard Log + RAG + Code` vs `TraceLog + RAG + Code`
- Ablation benchmark: `Standard Log + RAG` vs `TraceLog + RAG`

The primary benchmark is closer to the actual product hypothesis: can the system diagnose live incidents more accurately and efficiently when it has both historical context and code? The ablation is a secondary experiment used to isolate the contribution of TraceLog formatting itself.

In [1]:
from pathlib import Path
from IPython.display import Markdown, display
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / 'tracelog').exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tracelog.eval.benchmarking import (
    BenchmarkConfig,
    dataset_status_rows,
    diagnosis_rows,
    failure_case_rows,
    final_verdict_markdown,
    inventory_rows,
    load_results,
    markdown_table,
    operational_rows,
    retrieval_rows,
    run_benchmark,
    split_rows,
)

BASE_DIR = PROJECT_ROOT / 'docs' / 'eval' / 'benchmark'
MODE = 'analysis'  # switch to 'run' to regenerate the benchmark
CONFIG = BenchmarkConfig(base_dir=BASE_DIR, top_k=3, overwrite=True)

if MODE == 'run':
    results = run_benchmark(CONFIG)
else:
    results = load_results(BASE_DIR)

display(Markdown(f"Loaded benchmark results generated at `{results['generated_at']}`"))

Loaded benchmark results generated at `2026-03-09T13:17:20.191086+00:00`

## 1. Experiment Configuration

This section shows the current execution mode and benchmark configuration. `analysis` reads existing artifacts, while `run` regenerates the dataset and reruns the full benchmark. `top_k` is the number of historical incidents retrieved during the retrieval stage.

In [2]:
display(Markdown('```json\n' + json.dumps(results['config'], indent=2) + '\n```'))

```json
{
  "base_dir": "docs/eval/benchmark",
  "top_k": 3,
  "diagnoser_model": "gpt-4o-mini",
  "judge_model": "gpt-4o-mini",
  "overwrite": true
}
```

## 2. Scenario Inventory

This table shows the incident inventory included in the benchmark. Each row is one incident and shows its scenario family, the ground-truth root cause function, and the surface error function. The root cause is the upstream origin of the bug, while the surface error is the point where the failure becomes visible.

In [3]:
display(Markdown(markdown_table(inventory_rows(results))))

| incident_id | family | split | root_cause_function | surface_error_function | error_type |
| --- | --- | --- | --- | --- | --- |
| checkout_coupon_flip_h1 | ecommerce_bulk_checkout | historical | normalize_coupon | charge_payment | ValueError |
| checkout_coupon_flip_h2 | ecommerce_bulk_checkout | historical | normalize_coupon | charge_payment | ValueError |
| checkout_coupon_flip_q1 | ecommerce_bulk_checkout | query | normalize_coupon | charge_payment | ValueError |
| warehouse_snapshot_leak_h1 | warehouse_sync_reservation | historical | sync_warehouse_snapshot | reserve_bin_stock | TimeoutError |
| warehouse_dedupe_collision_h1 | warehouse_sync_reservation | historical | build_dedupe_key | reserve_bin_stock | TimeoutError |
| warehouse_snapshot_leak_q1 | warehouse_sync_reservation | query | sync_warehouse_snapshot | reserve_bin_stock | TimeoutError |
| gateway_refresh_order_h1 | api_gateway_audit | historical | verify_token | execute_business_action | PermissionError |
| gateway_refresh_order_h2 | api_gateway_audit | historical | verify_token | execute_business_action | PermissionError |
| gateway_tenant_cache_h1 | api_gateway_audit | historical | load_profile | execute_business_action | PermissionError |
| gateway_refresh_order_q1 | api_gateway_audit | query | verify_token | execute_business_action | PermissionError |

## 3. Dataset Generation Status

This section checks whether the required artifacts were generated for each incident. If `standard_log`, `tracelog_dump`, `aggregated_trace`, and `truth` are all `True`, that incident is fully available for the benchmark.

In [4]:
display(Markdown(markdown_table(dataset_status_rows(results))))

| incident_id | standard_log | tracelog_dump | aggregated_trace | truth |
| --- | --- | --- | --- | --- |
| checkout_coupon_flip_h1 | True | True | True | True |
| checkout_coupon_flip_h2 | True | True | True | True |
| checkout_coupon_flip_q1 | True | True | True | True |
| warehouse_snapshot_leak_h1 | True | True | True | True |
| warehouse_dedupe_collision_h1 | True | True | True | True |
| warehouse_snapshot_leak_q1 | True | True | True | True |
| gateway_refresh_order_h1 | True | True | True | True |
| gateway_refresh_order_h2 | True | True | True | True |
| gateway_tenant_cache_h1 | True | True | True | True |
| gateway_refresh_order_q1 | True | True | True | True |

## 4. Historical / Query Split Check

This table shows how the retrieval corpus and holdout queries are separated. `historical` incidents are searchable corpus items, and `query` incidents are diagnosis targets. If these splits leak into each other, the benchmark becomes invalid.

In [5]:
display(Markdown(markdown_table(split_rows(results))))

| split | count | families |
| --- | --- | --- |
| historical | 7 | api_gateway_audit, ecommerce_bulk_checkout, warehouse_sync_reservation |
| query | 3 | api_gateway_audit, ecommerce_bulk_checkout, warehouse_sync_reservation |

## 5. Retrieval Evaluation (Ablation Lens)

This section measures retrieval quality itself. At the moment it is interpreted as an ablation lens for the logs-only comparison. `SameRootCauseHit@K` means the top K results contain an incident with the same root-cause family. `MRR` measures how early a relevant incident appears, and `nDCG@3` measures ranking quality within the top 3 results.

In [6]:
display(Markdown(markdown_table(retrieval_rows(results))))

| condition | SameRootCauseHit@1 | SameRootCauseHit@3 | MRR | nDCG@3 |
| --- | --- | --- | --- | --- |
| Standard Log + RAG | 1.0 | 1.0 | 1.0 | 1.0 |
| TraceLog + RAG | 1.0 | 1.0 | 1.0 | 0.9732 |

## 6. Diagnosis Evaluation (Primary First)

This is the main table. It shows the code-aware primary benchmark first, then the logs-only ablation. `root_cause_accuracy` measures whether the upstream origin was identified correctly, and `surface_accuracy` measures whether the visible failure point was identified correctly. `evidence_match` and `actionability` are Judge-assigned quality scores in the 0 to 1 range.

In [7]:
display(Markdown(markdown_table(diagnosis_rows(results))))

| condition | root_cause_accuracy | surface_accuracy | evidence_match | actionability |
| --- | --- | --- | --- | --- |
| Standard Log + RAG + Code | 0.3333 | 1.0 | 0.65 | 0.5333 |
| TraceLog + RAG + Code | 0.6667 | 1.0 | 0.6 | 0.7333 |
| Standard Log + RAG | 0.0 | 0.0 | 0.5 | 0.3667 |
| TraceLog + RAG | 0.0 | 1.0 | 0.5667 | 0.6667 |

## 7. Failure Case Review

This section collects only the cases where the Judge marked the root cause as incorrect. It helps show whether the model stayed near the surface error or successfully traced the issue back to the upstream state origin.

In [8]:
display(Markdown(markdown_table(failure_case_rows(results))))

| incident_id | condition | predicted_root_cause | expected_root_cause | judge_reason |
| --- | --- | --- | --- | --- |
| checkout_coupon_flip_q1 | Standard Log + RAG | charge_payment | normalize_coupon | The analyst incorrectly identified 'charge_payment' as the root cause function instead of 'normalize_coupon'. The surface error function is also misidentified as 'run_incident' rather than 'charge_payment'. While there is some grounding in the evidence provided, it does not fully align with the expected evidence markers from the sealed truth, leading to a lower actionability score. |
| checkout_coupon_flip_q1 | TraceLog + RAG | ECommerceBulkCheckoutScenario.price_cart | normalize_coupon | The analyst incorrectly identified the root cause function as 'price_cart' instead of 'normalize_coupon', which is the actual function responsible for the error. However, the surface error function 'charge_payment' is correctly identified. The evidence grounding is moderate as some evidence markers align, but not all. Actionability is relatively high since the analyst provides a clear expected fix region. |
| checkout_coupon_flip_q1 | Standard Log + RAG + Code | price_cart | normalize_coupon | The analyst incorrectly identified the root cause function as 'price_cart' instead of 'normalize_coupon', which is the actual function responsible for the error. However, the surface error function 'charge_payment' is correctly identified. The evidence grounding is moderate as some evidence markers relate to the error but do not fully align with the expected evidence markers from the sealed truth. Actionability is low due to the misidentification of the root cause function, which affects the ability to implement a fix. |
| checkout_coupon_flip_q1 | TraceLog + RAG + Code | ECommerceBulkCheckoutScenario.price_cart | normalize_coupon | The analyst incorrectly identified the root cause function as 'price_cart' instead of 'normalize_coupon', which is the actual function responsible for the error. However, the surface error function 'charge_payment' is correctly identified. The evidence grounding is moderate as some evidence markers align, but not all. Actionability is relatively high since the analyst's diagnosis points to a specific function that can be fixed. |
| warehouse_snapshot_leak_q1 | Standard Log + RAG | reserve_bin_stock | sync_warehouse_snapshot | The analyst incorrectly identified 'reserve_bin_stock' as the root cause function instead of 'sync_warehouse_snapshot', which is responsible for the snapshot version leak. The surface error function is also misidentified; the expected error surface chain indicates that 'reserve_bin_stock' is not the initial function causing the timeout. The evidence provided does not fully align with the expected evidence markers, and the expected fix region differs from the sealed truth. |
| warehouse_snapshot_leak_q1 | TraceLog + RAG | WarehouseSyncScenario.reconcile_delta | sync_warehouse_snapshot | The analyst incorrectly identified the root cause function as 'reconcile_delta' instead of the correct 'sync_warehouse_snapshot'. However, the surface error function 'reserve_bin_stock' is correctly identified. The evidence grounding is moderate as some evidence markers align with the expected ones, but not all. Actionability is relatively high since the analyst's diagnosis provides a clear path to address the timeout error. |
| gateway_refresh_order_q1 | Standard Log + RAG | tracelog.eval.gateway_refresh_order_q1.standard | verify_token | The analyst's diagnosis identifies a PermissionError but attributes it to incorrect tenant profile handling rather than the expected root cause function 'verify_token'. The evidence markers provided do not align closely with the expected evidence markers from the sealed truth, leading to a lower grounding score. The actionability score is also reduced due to the mismatch in the expected fix region. |
| gateway_refresh_order_q1 | TraceLog + RAG | ApiGatewayAuditScenario.load_profile | verify_token | The analyst incorrectly identified the root cause function as 'ApiGatewayAuditScenario.load_profile' instead of 'verify_token'. However, the surface error function 'execute_business_action' is correctly identified. The evidence grounding is moderate as some evidence markers align with the expected ones, but not all. Actionability is somewhat low due to the misidentification of the root cause function, which may lead to ineffective fixes. |
| gateway_refresh_order_q1 | Standard Log + RAG + Code | load_profile | verify_token | The analyst incorrectly identified the root cause function as 'load_profile' instead of the correct 'verify_token'. However, the surface error function 'execute_business_action' is correctly identified. The evidence markers provided by the analyst align with the expected evidence markers, but the root cause identification is flawed, affecting the overall accuracy. |

## 8. Token / Latency / Cost Summary

This section measures operational efficiency. `input_tokens`, `output_tokens`, and `total_tokens` are the basic inputs for LLM cost estimation. `retrieval_latency`, `diagnosis_latency`, and `time_to_verdict` measure response speed during incident handling. Even if accuracy improves, the system may still be hard to adopt if the operational cost becomes too high.

In [9]:
display(Markdown(markdown_table(operational_rows(results))))

| condition | input_tokens | output_tokens | total_tokens | retrieval_latency | diagnosis_latency | time_to_verdict |
| --- | --- | --- | --- | --- | --- | --- |
| Standard Log + RAG + Code | 3861.67 | 184.0 | 4045.67 | 0.0663 | 4.6404 | 4.7067 |
| TraceLog + RAG + Code | 4773.67 | 178.33 | 4952.0 | 0.0652 | 4.242 | 4.3072 |
| Standard Log + RAG | 2850.33 | 181.33 | 3031.67 | 0.0574 | 4.9269 | 4.9843 |
| TraceLog + RAG | 3762.33 | 201.33 | 3963.67 | 0.0569 | 4.6645 | 4.7214 |

## 9. Final Verdict

The final conclusion summarizes the primary benchmark and the ablation benchmark separately. In other words, it distinguishes improvements under the product-facing `logs+code` condition from formatting effects observed in the secondary `logs-only` experiment.

In [10]:
display(Markdown(final_verdict_markdown(results)))

## Final Verdict

### Primary Benchmark
- Primary condition: `Standard Log + RAG + Code` vs `TraceLog + RAG + Code`
- Primary benchmark pass: `False`
- TraceLog root cause accuracy delta: `0.333`
- TraceLog surface accuracy delta: `0.000`
- Primary improved families: `api_gateway_audit`

### Ablation
- Ablation condition: `Standard Log + RAG` vs `TraceLog + RAG`
- Logs-only ablation pass: `False`
- TraceLog root cause accuracy delta: `0.000`
- TraceLog SameRootCauseHit@3 delta: `0.000`
- Ablation improved families: `none`

The primary benchmark is now the code-aware comparison, because that better matches the product goal of diagnosing live incidents with historical context and relevant code. The logs-only comparison remains valuable, but only as an ablation on formatting and retrieval behavior.

This notebook now treats the code-aware path as the primary product benchmark.
The logs-only comparison remains as an ablation to isolate formatting effects.